In [1]:
import torch
import torch.nn as nn
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert hidden_size % num_heads == 0
        
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        # 定义线性层
        self.w_q = nn.Linear(hidden_size, hidden_size)
        self.w_k = nn.Linear(hidden_size, hidden_size)
        self.w_v = nn.Linear(hidden_size, hidden_size)
        self.w_o = nn.Linear(hidden_size, hidden_size)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 1. 线性投影 + 分头 (Split Heads)
        # view: (B, L, D) -> (B, L, H, head_dim)
        # transpose: (B, L, H, head_dim) -> (B, H, L, head_dim)
        Q = self.w_q(q).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(k).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(v).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # 2. 缩放点积注意力 (Scaled Dot-Product Attention)
        # Q @ K.T -> (B, H, L, head_dim) @ (B, H, head_dim, L) -> (B, H, L, L)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        print(f"score shape:", scores.shape)
        
        # 如果有 mask (例如 padding mask 或 causal mask)
        if mask is not None:
            # mask 为 0 的位置填入极小值 (比如 -1e9)，让 softmax 后概率接近 0
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = torch.softmax(scores, dim=-1)
        
        # weighted sum -> (B, H, L, L) @ (B, H, L, head_dim) -> (B, H, L, head_dim)
        context = torch.matmul(attn, V)

        # 3. 拼接 (Concat) + 线性变换
        # transpose: (B, H, L, head_dim) -> (B, L, H, head_dim)
        # contiguous + view: -> (B, L, hidden_size)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.hidden_size)
        
        output = self.w_o(context)
        
        return output

In [ ]:
# 1. 定义超参数
BATCH_SIZE = 2
SEQ_LEN = 10
HIDDEN_SIZE = 64  # 必须能被 NUM_HEADS 整除
NUM_HEADS = 4

print(f"Testing MHA with: B={BATCH_SIZE}, L={SEQ_LEN}, D={HIDDEN_SIZE}, H={NUM_HEADS}")

# 2. 实例化模型
mha = MultiHeadAttention(hidden_size=HIDDEN_SIZE, num_heads=NUM_HEADS)

# 3. 构造随机输入 (模拟 Self-Attention: Q=K=V)
# 输入形状: (Batch_Size, Seq_Len, D_Model)
x = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_SIZE)

# 4. 前向传播测试 - 无 Mask
try:
    output = mha(q=x, k=x, v=x)
    print(f"\n[Pass] Forward output shape: {output.shape}")
    
    # 验证输出形状是否与输入一致
    expected_shape = (BATCH_SIZE, SEQ_LEN, HIDDEN_SIZE)
    assert output.shape == expected_shape, f"Expected {expected_shape}, got {output.shape}"
    
except Exception as e:
    print(f"\n[Fail] Forward pass error: {e}")

# 5. 前向传播测试 - 带 Mask
# Mask 通常形状为 (Batch, 1, 1, Seq_Len) 或 (Batch, 1, Seq_Len, Seq_Len)
# 这里模拟一个简单的 causal mask (下三角)
mask = torch.tril(torch.ones(SEQ_LEN, SEQ_LEN)).view(1, 1, SEQ_LEN, SEQ_LEN)

try:
    output_masked = mha(q=x, k=x, v=x, mask=mask)
    print(f"[Pass] Masked forward output shape: {output_masked.shape}")
    assert output_masked.shape == expected_shape
except Exception as e:
    print(f"[Fail] Masked forward pass error: {e}")

print("\n✅ All tests passed!")

Testing MHA with: B=2, L=10, D=64, H=4
score shape: torch.Size([2, 4, 10, 10])

[Pass] Forward output shape: torch.Size([2, 10, 64])
score shape: torch.Size([2, 4, 10, 10])
[Pass] Masked forward output shape: torch.Size([2, 10, 64])

✅ All tests passed!


: 